# 03: Model Training & Experiment Tracking

## Project: AgroVision
**Purpose:** Train a convolutional neural network (CNN) to classify fruit freshness. Use **MLflow** to track hyperparameters, metrics, and model artifacts for reproducibility.

### Key Objectives
1.  **Data Ingestion:** Load the cleaned dataset (`dataset_cleaned.csv`) and normalization statistics from the EDA phase.
2.  **Pipeline Construction:** Implement a custom PyTorch `Dataset` and `DataLoaders` with the computed transformations.
3.  **Experiment Tracking:** Initialize an MLflow experiment to log parameters (learning rate, batch size) and metrics (loss, accuracy).
4.  **Model Training:** Fine-tune a pre-trained architecture (e.g., ResNet18) using the computed class weights to handle imbalance.
5.  **Artifact Storage:** Save the best model state and training logs for future inference.

In [ ]:
import json
from pathlib import Path

import mlflow
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

## 1. Environment & Configuration
Define project paths, compute device (GPU/CPU), and initialize the MLflow tracking URI.

In [ ]:
PROJECT_ROOT_DIR_PATH = Path.cwd().parent

DATA_DIR_PATH = PROJECT_ROOT_DIR_PATH / "data"
RAW_DATA_DIR_PATH = DATA_DIR_PATH / "raw"
PROCESSED_DATA_DIR_PATH = DATA_DIR_PATH / "processed"
DATASET_CLEAN_FILE_PATH = PROCESSED_DATA_DIR_PATH / "dataset_clean.csv"

ARTIFACTS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "artifacts"
CLASS_WEIGHTS_FILE_PATH = ARTIFACTS_DIR_PATH / "class_weights.json"
NORMALIZATION_STATS_FILE_PATH = ARTIFACTS_DIR_PATH / "normalization_stats.json"

MODELS_DIR_PATH = PROJECT_ROOT_DIR_PATH / "models"
MODELS_CHECKPOINTS_DIR_PATH = MODELS_DIR_PATH / "checkpoints"
BEST_MODEL_FILE_PATH = MODELS_CHECKPOINTS_DIR_PATH / "best_model.pt"

MLFLOW_SERVER_URI = "http://0.0.0.0:5000"
MLFLOW_EXPERIMENT_NAME = "agrovision_training"

mlflow.set_tracking_uri(MLFLOW_SERVER_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {DEVICE}")

## 2. Load Metadata & Statistics
Import the processed artifacts: the cleaned dataset index, class weights for the loss function, and normalization stats for image preprocessing.

In [ ]:
df = pd.read_csv(DATASET_CLEAN_FILE_PATH)

classes = sorted(df["label"].unique())
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}

with open(NORMALIZATION_STATS_FILE_PATH) as f:
    stats = json.load(f)
    mean = stats["mean"]
    std = stats["std"]

with open(CLASS_WEIGHTS_FILE_PATH) as f:
    weights_dict = json.load(f)
    weights_list = [weights_dict[cls] for cls in classes]
    class_weights_tensor = torch.tensor(weights_list, dtype=torch.float).to(DEVICE)

print(f"Dataset size: {len(df)}")
print(f"Classes: {classes}")
print(f"Normalization: mean={mean}, std={std}")

## 3. Custom Dataset Definition
Implement the `AgroVisionDataset` class to handle image loading, label encoding, and dynamic transformations (augmentation + normalization).

In [ ]:
class AgroVisionDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self.root_dir / row["filepath"]
        image = Image.open(img_path).convert("RGB")

        label_str = row["label"]
        label = class_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

## 4. Transforms & DataLoaders
Prepare the training and validation pipelines. We apply data augmentation (flip, rotation) only to the training set to improve generalization.

In [ ]:
data_transforms = {
    "train": transforms.Compose(
        [
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    ),
    "val": transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ]
    ),
}

train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "val"]

datasets = {
    "train": AgroVisionDataset(
        train_df, RAW_DATA_DIR_PATH, transform=data_transforms["train"]
    ),
    "val": AgroVisionDataset(
        val_df, RAW_DATA_DIR_PATH, transform=data_transforms["val"]
    ),
}

BATCH_SIZE = 32
dataloaders = {
    "train": DataLoader(
        datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    ),
    "val": DataLoader(
        datasets["val"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    ),
}

print(f"Train batches: {len(dataloaders['train'])}")

## 5. Training Loop with MLflow
Execute the training process. This block encapsulates the entire run within an `mlflow.start_run()` context to log:

1.  **Parameters:** Learning rate, epochs, architecture.
2.  **Metrics:** Train loss, validation accuracy (per epoch).
3.  **Model:** The final model state dictionary.

In [ ]:
def train_model(hyperparams):
    with mlflow.start_run(run_name=hyperparams["run_name"]):
        mlflow.log_params(hyperparams)

        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, len(classes))
        model = model.to(DEVICE)

        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
        optimizer = optim.Adam(model.parameters(), lr=hyperparams["lr"])

        best_acc = 0.0

        print(f"Starting Run: {hyperparams['run_name']}")

        for epoch in range(hyperparams["epochs"]):
            model.train()
            running_loss = 0.0

            pbar_train = tqdm(
                dataloaders["train"],
                desc=f"Epoch {epoch + 1}/{hyperparams['epochs']} [Train]",
                leave=False,
            )

            for inputs, labels in pbar_train:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                pbar_train.set_postfix(loss=loss.item())

            epoch_loss = running_loss / len(datasets["train"])

            model.eval()
            running_corrects = 0

            pbar_val = tqdm(
                dataloaders["val"],
                desc=f"Epoch {epoch + 1}/{hyperparams['epochs']} [Val]",
                leave=False,
            )

            with torch.no_grad():
                for inputs, labels in pbar_val:
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    running_corrects += torch.sum(preds == labels.data)

            epoch_acc = running_corrects.double() / len(datasets["val"])

            print(
                f"Epoch {epoch + 1}: Train Loss = {epoch_loss:.4f} | Val Acc = {epoch_acc:.4f}"
            )

            mlflow.log_metric("train_loss", epoch_loss, step=epoch)
            mlflow.log_metric("val_accuracy", epoch_acc.item(), step=epoch)

            if epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), BEST_MODEL_FILE_PATH)

        mlflow.pytorch.log_model(model, name="model_fruit_freshness")
        print(f"Training Complete. Best Val Acc: {best_acc:.4f}")


PARAMS = {
    "run_name": "resnet18_baseline_v1",
    "epochs": 10,
    "lr": 0.001,
    "batch_size": BATCH_SIZE,
    "architecture": "resnet18",
}

train_model(PARAMS)

## 6. Summary & Next Steps
The training pipeline has been successfully executed. A ResNet18 baseline has been fine-tuned on the AgroVision dataset, with all experiments tracked and versioned via MLflow.

**Key Outcomes:**
1.  **Model Trained:** A ResNet18 architecture was successfully fine-tuned for fruit freshness classification.
2.  **Experiment Logged:** Training dynamics (loss, accuracy) and hyperparameters were recorded in the MLflow tracking database.
3.  **Artifacts Saved:** The best-performing model weights (`best_model.pt`) have been serialized and stored in the `mlruns/checkpoints` directory.

**Next Steps:**
1.  **Model Auditing:** Evaluation on the hold-out **test set** is required to verify real-world generalization.
2.  **Performance Metrics:** The best-performing model will be loaded in **`04_model_evaluation.ipynb`** to generate the confusion matrix and classification report.
3.  **Error Analysis:** Specific misclassifications will be analyzed to identify potential model weaknesses before deployment.